# CS1 EXP-2 -- SVD(TF-IDF) + Static Features -> MLP (rotating 5-fold, staged nested search)

## 1. Workspace root

In [1]:
from pathlib import Path

WORKSPACE_ROOT = Path.cwd()

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)


WORKSPACE_ROOT: /workspace


## 2. Runtime switches

Run profile first. After the profile succeeds, disable profile and enable the official development CV.


In [ ]:
RUN_PROFILE_FOLD = False
RUN_OFFICIAL = True
REQUIRE_GPU_FOR_OFFICIAL_RUN = True

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"
MODEL_RANDOM_STATE = 42

#CODE_COLUMN = "normalized_code"
CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

print("RUN_PROFILE_FOLD:", RUN_PROFILE_FOLD)
print("RUN_OFFICIAL:", RUN_OFFICIAL)
print("CODE_COLUMN:", CODE_COLUMN)


## 3. Define paths


In [ ]:
from pathlib import Path

DRIVE_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"
STATIC_FEATURE_DIR = PROCESSED_DIR / "static_features_downsampled20k"
STATIC_FEATURE_PATH = STATIC_FEATURE_DIR / "cs1_static_features_v1.parquet"

EXP2_OUTPUT_DIR = OUTPUT_ROOT / f"exp2_mlp_rotating5fold_{CODE_COLUMN_TAG}"

for directory in [PROCESSED_DIR, MANIFEST_ROOT, OUTPUT_ROOT, STATIC_FEATURE_DIR, EXP2_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Downsampled parquet:", DOWNSAMPLED_PARQUET)
print("Manifest:", MANIFEST_PATH)
print("Static feature cache:", STATIC_FEATURE_PATH)
print("Output dir:", EXP2_OUTPUT_DIR)


## 4. Clone or refresh the repository branch


In [4]:
from pathlib import Path
import sys

REPO_DIR = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

import urllib.request
import zipfile


def download_and_extract_repo(repo_url, branch, target_dir):
    if target_dir.exists():
        print(f"Repository already exists at {target_dir}")
        return

    print(f"Downloading {repo_url} (branch: {branch}) without git...")
    clean_url = repo_url.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{branch}.zip"
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target_dir.parent / f"{target_dir.name}_download_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir.parent)

    repo_name = clean_url.split("/")[-1]
    extracted_folder = target_dir.parent / f"{repo_name}-{branch}"
    if extracted_folder.exists():
        extracted_folder.rename(target_dir)

    zip_path.unlink()
    print(f"Repository ready at {target_dir}")

download_and_extract_repo(REPO_URL, REPO_BRANCH, REPO_DIR)

if not SRC_DIR.exists():
    raise FileNotFoundError(f"Expected source directory does not exist: {SRC_DIR}")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source path:", SRC_DIR)


Repository already exists at /workspace/DiverseVul--IS-Project
Repository: /workspace/DiverseVul--IS-Project
Source path: /workspace/DiverseVul--IS-Project/vuln-detection/src


## 5. Verify required repository files


In [ ]:
required_repo_files = [
    SRC_DIR / "utils" / "evaluation.py",
    SRC_DIR / "utils" / "split_manifest.py",
    SRC_DIR / "case_study_1" / "static_features.py",
    SRC_DIR / "case_study_1" / "exp2" / "exp2_mlp.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError("Required EXP-2 files are missing:\n" + "\n".join(missing_repo_files))

print("Required Case Study 1 files are present.")


## 6. Install dependencies


In [6]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "pyyaml",
        "pyarrow",
        "joblib",
    ],
    check=True,
)
print("Dependencies installed or already available.")


Dependencies installed or already available.


## 7. Import modules and check accelerator


In [ ]:
import importlib
import json
import subprocess
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

exp2_mlp = importlib.import_module("case_study_1.exp2.exp2_mlp")
static_features = importlib.import_module("case_study_1.static_features")
split_manifest = importlib.import_module("utils.split_manifest")
evaluation = importlib.import_module("utils.evaluation")

required_exp2_api = ["EXP2_VERSION", "Exp2Config", "run_exp2_profile_fold", "run_exp2"]
missing_exp2_api = [name for name in required_exp2_api if not hasattr(exp2_mlp, name)]
if missing_exp2_api:
    raise AttributeError(f"EXP-2 runner is missing API: {missing_exp2_api}")

print("EXP-2 runner:", exp2_mlp.__file__)
print("EXP-2 version:", exp2_mlp.EXP2_VERSION)
print("Static feature count:", len(static_features.FEATURE_COLUMNS))
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 8. Load the downsampled dataset, shared 5-fold manifest, and static features

In [ ]:
if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(
        f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}\n"
        "Run notebooks/scope2_preprocessing.ipynb first to build it."
    )
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Missing manifest: {MANIFEST_PATH}\n"
        "Run notebooks/scope2_preprocessing.ipynb's manifest-generation section first."
    )

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

required_full_columns = {"source_row_id", "code", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_full_columns = required_full_columns.difference(full_df.columns)
if missing_full_columns:
    raise KeyError(f"Downsampled dataset missing columns: {sorted(missing_full_columns)}")

if STATIC_FEATURE_PATH.is_file():
    static_df = pd.read_parquet(STATIC_FEATURE_PATH)
    print("Loaded cached static features:", STATIC_FEATURE_PATH)
else:
    print("Computing static features for the downsampled dataset (deterministic, no leakage)...")
    static_config = static_features.StaticFeatureConfig(source_id_column="source_row_id", code_column="code")
    static_df = static_features.extract_static_feature_frame(full_df, config=static_config)
    static_features.save_static_feature_artifacts(
        static_df, STATIC_FEATURE_DIR, config=static_config, source_dataset_path=DOWNSAMPLED_PARQUET
    )
    print("Saved static features to:", STATIC_FEATURE_PATH)

print("Downsampled dataset:", full_df.shape)
print("Manifest:", manifest_df.shape)
print("Static features:", static_df.shape)
print(manifest_df["fold"].value_counts().sort_index())


## 9. Manifest validation and empty-code guard

In [ ]:
if full_df["source_row_id"].duplicated().any():
    raise RuntimeError("full_df contains duplicate source_row_id values.")
if manifest_df["source_row_id"].duplicated().any():
    raise RuntimeError("manifest_df contains duplicate source_row_id values.")
if static_df["source_row_id"].duplicated().any():
    raise RuntimeError("static_df contains duplicate source_row_id values.")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

joined = manifest_df.set_index("source_row_id").join(full_indexed[["label", "project"]], rsuffix="_dataset")
if not (joined["label"].astype(int) == joined["label_dataset"].astype(int)).all():
    raise RuntimeError("Label mismatch between downsampled dataset and manifest.")
if not (joined["project"].astype(str) == joined["project_dataset"].astype(str)).all():
    raise RuntimeError("Project mismatch between downsampled dataset and manifest.")

static_ids = set(static_df["source_row_id"].tolist())
if static_ids != manifest_ids:
    raise RuntimeError(
        "Static feature cache coverage does not match the dataset exactly. "
        f"Missing={len(manifest_ids - static_ids)}, extra={len(static_ids - manifest_ids)}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].copy().reset_index(drop=True)
dataset_frame = dataset_frame[
    ["source_row_id", "code", "abstracted_code_v1", "normalized_code", "label", "project"]
].copy()

print("Dataset ready:", dataset_frame.shape, "| positive rate:", dataset_frame["label"].mean())
print("Unique projects:", dataset_frame["project"].nunique())


## 10. Empty-code guard

In [ ]:
EMPTY_CODE_SENTINEL = "EMPTY_ABSTRACTED_CODE_SAMPLE"
dataset_frame[CODE_COLUMN] = dataset_frame[CODE_COLUMN].fillna("").astype(str)
empty_mask = dataset_frame[CODE_COLUMN].str.strip().eq("")
n_empty = int(empty_mask.sum())
if n_empty:
    dataset_frame.loc[empty_mask, CODE_COLUMN] = EMPTY_CODE_SENTINEL
    print(f"Replaced {n_empty} empty {CODE_COLUMN} rows with sentinel token.")
assert not dataset_frame[CODE_COLUMN].str.strip().eq("").any()
print("Empty-code guard passed.")


## 11. Configure EXP-2

In [ ]:
config = exp2_mlp.Exp2Config(
    experiment_name=f"cs1_exp2_mlp_{CODE_COLUMN_TAG}",
    code_column=CODE_COLUMN,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
    fold_column="fold",
    n_splits=5,
    random_state=MODEL_RANDOM_STATE,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)

print("Code column:", config.code_column)
print("word_max_features_grid:", config.word_max_features_grid)
print("char_max_features_grid:", config.char_max_features_grid)
print("svd_n_components_grid:", config.svd_n_components_grid)
print("hidden_dim_1_grid:", config.hidden_dim_1_grid)
print("hidden_dim_2_grid:", config.hidden_dim_2_grid)
print("learning_rate_grid:", config.learning_rate_grid)
print("Device:", config.device)
print("Output directory:", EXP2_OUTPUT_DIR)


## 12. Optional profile run -- staged inner search + refit for one outer fold

In [ ]:
if RUN_PROFILE_FOLD:
    profile_start = time.perf_counter()
    profile = exp2_mlp.run_exp2_profile_fold(
        normalized_frame=dataset_frame,
        static_features_frame=static_df,
        manifest=manifest_df,
        fold_id=4,
        config=config,
    )
    print("Profile duration minutes:", (time.perf_counter() - profile_start) / 60)
    print("Selected hyperparameters:")
    display(pd.DataFrame([profile["selection"]]))
    print("Fold metrics:")
    display(profile["profile_metrics"])
else:
    print("RUN_PROFILE_FOLD=False; skipping profile.")


## 13. Inspect profile result

In [ ]:
if "profile" not in globals():
    print("No profile result in memory. Run the profile cell first or skip this section.")
else:
    print("Inner search summary (Stage A then Stage B):")
    display(profile["inner_search"])
    print("\nFold training metadata:")
    display(profile["training_metadata"])


## 14. Official rotating 5-fold run

In [ ]:
def _resolve_repo_commit(repo_root: Path, repo_branch: str) -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_root), "rev-parse", "HEAD"], stderr=subprocess.DEVNULL,
        ).decode().strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return f"unknown (repo fetched via zip archive, branch={repo_branch}, no .git metadata)"


if RUN_OFFICIAL:
    if REQUIRE_GPU_FOR_OFFICIAL_RUN and not torch.cuda.is_available():
        raise RuntimeError(
            "GPU is required for this official MLP run, but torch.cuda.is_available() is False. "
            "Set REQUIRE_GPU_FOR_OFFICIAL_RUN=False in the runtime-switches cell to accept a much slower CPU run."
        )

    results = exp2_mlp.run_exp2(
        normalized_frame=dataset_frame,
        static_features_frame=static_df,
        manifest=manifest_df,
        config=config,
        output_dir=EXP2_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "input_column": CODE_COLUMN,
            "manifest_path": str(MANIFEST_PATH),
            "static_feature_cache_path": str(STATIC_FEATURE_PATH),
            "repo_commit": _resolve_repo_commit(REPO_DIR, REPO_BRANCH),
        },
    )
    print("\nOfficial EXP-2 run complete.")
else:
    results = None
    print("RUN_OFFICIAL=False; official training skipped.")


## 15. Display result summary

In [ ]:
if results is None:
    print("No official results object in memory. Set RUN_OFFICIAL=True.")
else:
    pooled = results["evaluation"]["pooled_metrics"]
    fold_metrics = results["evaluation"]["fold_metrics"]
    fold_summary = results["evaluation"]["fold_summary"]
    fold_training = results["fold_training"]

    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in pooled.items()]))

    print("Per-fold metrics:")
    display(fold_metrics)

    print("Mean +/- std across the 5 outer folds (headline result):")
    display(fold_summary)

    print("Selected hyperparameters by outer fold:")
    display(fold_training[[
        "fold", "word_max_features", "char_max_features", "word_min_df", "char_min_df",
        "svd_n_components", "hidden_dim_1", "hidden_dim_2", "learning_rate", "decision_threshold",
    ]])

    print("Artifacts:", results.get("artifacts"))


## 15b. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
import utils.confidence_intervals as confidence_intervals

exp2_oof_ci = confidence_intervals.bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(confidence_intervals.format_ci_report(exp2_oof_ci))


## 16. Error analysis: pooled OOF false positives/negatives

In [ ]:
oof = results["oof_predictions"].merge(dataset_frame[["source_row_id", "code"]], on="source_row_id", how="left")

false_positives = oof[(oof["label"] == 0) & (oof["y_pred"] == 1)]
false_negatives = oof[(oof["label"] == 1) & (oof["y_pred"] == 0)]

print(f"Extracted {len(false_positives)} False Positives and {len(false_negatives)} False Negatives (pooled OOF, full dataset).")

sample_columns = ["source_row_id", "project", "y_score", "code"]
false_positives[sample_columns].sample(n=min(5, len(false_positives)), random_state=42).to_csv(
    EXP2_OUTPUT_DIR / "sample_false_positives.csv", index=False
)
false_negatives[sample_columns].sample(n=min(5, len(false_negatives)), random_state=42).to_csv(
    EXP2_OUTPUT_DIR / "sample_false_negatives.csv", index=False
)
